## Trying out compiling the information as a tensor ##

In [3]:
from lidar_object_tracking.dataset import unpack_from_pickle, read_data_json, load_json_data
import tensorflow as tf
import numpy as np
import pandas as pd

from lidar_object_tracking.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

2026-03-02 15:34:54.830 | INFO     | lidar_object_tracking.config:<module>:11 - PROJ_ROOT path is: /home/danielb/Desktop/CodingProjects/lidar_object_tracking


2026-03-02 15:34:55.403217: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-03-02 15:34:55.444688: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-03-02 15:34:56.617660: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [40]:
data_dict = load_json_data(PROCESSED_DATA_DIR,'004')
i = 0

tensor = np.zeros((40000,4))
for key in range(80):
    frame_numpy_data = []
    numpy_data = data_dict[key].to_numpy()
    for datum in numpy_data:
        frame_datum = np.insert(datum, 0, key*0.1)
        tensor[i] = frame_datum
        i += 1

tensor

array([[ 0.00000000e+00, -7.12199986e+00,  9.27849147e+00,
         5.11409035e-01],
       [ 0.00000000e+00, -7.02555857e+00,  9.22475662e+00,
         5.21977662e-01],
       [ 0.00000000e+00, -5.81751224e+00,  9.83252011e+00,
         1.15202922e+00],
       ...,
       [ 7.90000000e+00,  1.64333547e-03,  6.32879398e+00,
         2.90373501e-01],
       [ 7.90000000e+00,  2.12884373e-02,  6.31893295e+00,
         2.91329443e-01],
       [ 7.90000000e+00,  5.08663090e-02,  6.31854783e+00,
         2.89125052e-01]], shape=(40000, 4))

In [110]:
from sklearn.cluster import DBSCAN

db = DBSCAN(eps = 1, min_samples = 50)

clusters = db.fit(tensor)

labels = clusters.labels_

np.unique_counts(labels)

UniqueCountsResult(values=array([-1,  0,  1,  2,  3,  4,  5]), counts=array([  125,  9464, 10047,  8868, 11225,   175,    96]))

In [120]:
filt = (labels == 0)

tensor_label = tensor[filt]

tensor_label_dict = {}
for value in np.unique(tensor_label[:,0]):
    tensor_label_dict[int(value*10)] = tensor_label[tensor_label[:,0] == value][:,[1,2,3]]

key_list = sorted(tensor_label_dict.keys())

for idx in range(min(key_list), max(key_list)+1):
    try:
        tensor_label_dict[idx]
    except KeyError:
        tensor_label_dict[idx] = tensor_label_dict[idx - 1]

sorted_tld = dict(sorted(tensor_label_dict.items()))

for key in sorted_tld.keys():
    sorted_tld[key] = np.concatenate((sorted_tld[key], [[-10,0,0],[10,15,5]]))

sorted_tld

{0: array([[-5.81751224e+00,  9.83252011e+00,  1.15202922e+00],
        [-6.45342887e+00,  9.19329047e+00,  7.37241009e-01],
        [-6.48150061e+00,  9.18571032e+00,  1.73553872e-01],
        ...,
        [-4.02765449e+00,  9.40002448e+00,  1.44939676e-02],
        [-1.00000000e+01,  0.00000000e+00,  0.00000000e+00],
        [ 1.00000000e+01,  1.50000000e+01,  5.00000000e+00]],
       shape=(352, 3)),
 1: array([[ -6.27583291,   9.24832963,   0.54679135],
        [ -6.25545683,   9.20918749,   0.39783382],
        [ -5.88480704,   9.17660972,   0.20685167],
        ...,
        [ -4.42207414,   9.32649796,   0.11724196],
        [-10.        ,   0.        ,   0.        ],
        [ 10.        ,  15.        ,   5.        ]], shape=(376, 3)),
 2: array([[ -5.6226258 ,   9.28223479,   0.02304225],
        [ -5.44887704,   9.18459586,   0.43245869],
        [ -5.86879679,  10.14357828,   0.83308541],
        ...,
        [ -4.26261246,   9.32448327,   0.12892094],
        [-10.        , 

In [121]:
import plotly.graph_objects as go

def plot_with_slider(dataset):
    fig = go.Figure()

    max_frame = max(dataset.keys())
    min_frame = min(dataset.keys())
    span_frame = (max_frame - min_frame)
    for i in range(min_frame, max_frame+1):

        fig.add_trace(
            go.Scatter3d(
                visible=False,
                mode="markers",
                marker=dict(size=2),
                x=dataset[i][:,0],
                y=dataset[i][:,1],
                z=dataset[i][:,2],
            )
        )
        fig.update_layout(height=500, width=750)
        fig.update_scenes(
            xaxis_range=[-10, 10],
            yaxis_range=[0, 15],
            zaxis_range=[0, 5],
            aspectmode="data",
            overwrite=True
        )

    fig.data[0].visible = True

    steps = []
    for i in range(len(list(fig.data))):
        step = dict(
            method="update",
            args=[{"visible": [False] * len(list(fig.data))}],
            label=f"{i / 10}s",
        )
        step["args"][0]["visible"][i] = True
        steps.append(step)

    sliders = [
        dict(
            active=0,
            currentvalue={
                "prefix": "Frame: ",
            },
            pad={"t": 80},
            steps=steps,
        )
    ]

    fig.update_layout(sliders=sliders)

    fig.show()



In [122]:
plot_with_slider(sorted_tld)